# A6 — Por que a data de internação é a âncora, e quem ficou de fora (ficha #81)

**Pessoa A.** Duas perguntas que a banca costuma fazer: *"por que a internação, e não a notificação?"* e *"quem ficou de fora da coorte?"* (viés de seleção).

Lê da base **só** `dt_notific`, `dt_interna`, `dt_sin_pri`, `evolucao`, `sg_uf`, `nu_idade_n`, `tp_idade`, `hospital` e `nosocomial`, e grava em `outputs/final/`:

| Tabela | Arquivo | Pergunta |
|---|---|---|
| 1. Âncora | `ancora_notificacao_internacao.csv` | dias entre notificação e internação (mediana e IQR), por ano e desfecho |
| 2. Quem ficou de fora | `excluidos_vs_incluidos.csv` | hospitalizados excluídos por desfecho não elegível × incluídos, por ano, região e faixa etária |
| 3. Casos hospitalares | `intervalo_negativo_nosocomial.csv` | entre as internações com intervalo negativo, quantas são nosocomiais, e o desfecho |

Cada tabela vem com **uma frase de leitura** para a seção 2 do relatório (`frases_secao2.md`).

In [ ]:
# ----- Setup (não editar) -----
import os, sys
from pathlib import Path

# PKG_ROOT: pasta que contém src/protocolo (onde o pacote foi copiado)
# REPO_ROOT: raiz do repositório (tem .git ou data/); é de onde lemos dados e gravamos saídas
_parents = [Path.cwd(), *Path.cwd().parents]
PKG_ROOT = next((p for p in _parents if (p / "src" / "protocolo").exists()), None)
if PKG_ROOT is None:
    raise FileNotFoundError(
        "Pasta src/protocolo não encontrada acima de " + str(Path.cwd()) + ". "
        "Copie 'src/protocolo/' do pacote para o 'src/' do repositório "
        "(deve existir src/protocolo/verificacoes.py) e rode a célula de novo.")
REPO_ROOT = next((p for p in _parents if (p / ".git").exists() or (p / "data").exists()), PKG_ROOT)
os.chdir(REPO_ROOT); sys.path.insert(0, str(PKG_ROOT))

# Proteção contra outro pacote chamado 'src' (ex.: instalado por pip) ou 'src' antigo em cache no kernel:
# um pacote regular em qualquer ponto do sys.path vence um 'src' sem __init__.py.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
_init = PKG_ROOT / "src" / "__init__.py"
if not _init.exists():
    _init.touch()
    print("Criado", _init, "(torna o src do projeto um pacote regular)")

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from src.protocolo import verificacoes as v
if getattr(v, "MODULE_VERSION", "0") < "1.3":
    raise RuntimeError(
        "src/protocolo/verificacoes.py está desatualizado (versão " + str(getattr(v, "MODULE_VERSION", "antiga")) + "). "
        "Substitua pelo arquivo do pacote novo e REINICIE o kernel (o Python mantém o módulo antigo em cache).")

# ----- Parâmetros -----
USAR_DEMO = False                       # True = dados sintéticos só para testar o notebook
DATA_PATH = None                        # None = procura Parquet/CSV em data/ ; ou informe o caminho do arquivo
SAMPLE = 0                              # ex.: 200_000 para execução rápida
OUT = "outputs/tables/pessoa_a"         # saídas (tabelas, atas, decisões), relativas à raiz do repositório
CFG_PATH = next((str(p) for p in [REPO_ROOT / "configs" / "decisoes.yaml", PKG_ROOT / "configs" / "decisoes.yaml"]
                 if p.exists()), "configs/decisoes.yaml")
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (alvo, features, coluna_tempo)
COLUNAS = ["dt_notific", "dt_interna", "dt_sin_pri", "evolucao", "sg_uf", "nu_idade_n", "tp_idade", "hospital", "nosocomial"]   # só estas (ficha #81)                   # "minimas" (menos memória) | "classificadas" | "todas" (pesado!) | lista de nomes
FINAL_DIR = "outputs/final"          # onde gravar as 3 tabelas e as frases (ficha #81)
print("Raiz do repositório:", REPO_ROOT)
print("Pacote (src/protocolo):", PKG_ROOT)
print("Config de decisões:", CFG_PATH)
cfg = v.load_config(CFG_PATH)
rep = v.Report(OUT, "relatorio_A6")
df = v.prepare(v.carregar_base(DATA_PATH, USAR_DEMO, SAMPLE, COLUNAS, cfg, SUPERVISED_CONFIG))
print(f"Memória do DataFrame: {v.uso_memoria_gb(df):.2f} GB")

## 1. Gerar as três tabelas e as frases

In [ ]:
res = v.gerar_tabelas_ancora_selecao(df, cfg, FINAL_DIR, rep)
for k, f in res["frases"].items():
    print(f"[{k}]\n{f}\n")

## 2. Tabela 1 — âncora
`dt_notific − dt_interna`: positivo = notificação **depois** da internação. A coluna `pct_ano_diferente` mostra quantas internações mudam de ano entre as duas datas (é o que misturaria períodos se a âncora fosse a notificação).

In [ ]:
t1 = res["ancora"]
display(t1[(t1["ano"] == "todos")])
display(t1[(t1["ano"] != "todos") & (t1["desfecho"] == "todos")])

## 3. Tabela 2 — quem ficou de fora
Excluído = hospitalizado com `evolucao` 3, 9 ou vazia; incluído = 1 ou 2. `pct_excluidos_no_nivel` é a taxa de exclusão dentro do nível; `pct_entre_excluidos` e `pct_entre_incluidos` comparam a **composição**; `diferenca_pp` é a diferença entre as duas (positiva = o nível está sobre-representado entre os excluídos).

In [ ]:
t2 = res["excluidos_vs_incluidos"]
display(t2[t2["dimensao"] == "geral"])
for dim in ("ano", "regiao", "faixa_etaria"):
    display(t2[t2["dimensao"] == dim].drop(columns=["dimensao"]))

## 4. Tabela 3 — intervalo negativo e casos hospitalares
Internações com início dos sintomas **depois** da internação. `nosocomial = 1` indica infecção adquirida no hospital (caso legítimo); `2` e `9`/vazio sugerem erro de data.

> **Para conciliar:** a decisão **D_COORTE_006** exclui esses casos da coorte, enquanto a proposta do #82 (A5) os mantém com valor ausente e `flag_intervalo_invalido`. Esta tabela mostra quantos são nosocomiais, base para decidir se a exclusão faz sentido.

In [ ]:
display(res["intervalo_negativo_nosocomial"])

## 5. Fechamento

In [ ]:
path = rep.write_md()
print("Relatório:", path)
print("Arquivos em", FINAL_DIR, ":", sorted(p.name for p in Path(FINAL_DIR).glob("*") if p.is_file()))

### Checklist (ficha #81)
- [ ] `ancora_notificacao_internacao.csv`, `excluidos_vs_incluidos.csv` e `intervalo_negativo_nosocomial.csv` em `outputs/final/`
- [ ] Uma frase de leitura por tabela copiada para o relatório (seção 2) a partir de `frases_secao2.md`
- [ ] Conferir a Tabela 2 contra o waterfall (97.786 e 36.923 da ficha): os totais por tipo de exclusão devem bater
- [ ] D_COORTE_006 × #82 conciliados com base na Tabela 3